# Guided project — three-run e-commerce pipeline

**Time:** 150 minutes. Run all cells in order. Every code cell is complete.

```mermaid
flowchart LR
 A[CSV / JSON / TSV] --> B[Explicit schema]
 B --> C[Deduplicate and safe-cast]
 C --> D{Quality valid?}
 D -- No --> Q[Rejected Parquet]
 D -- Yes --> E[Completed sales]
 E --> F[Broadcast reference joins]
 F --> G{Keys matched?}
 G -- No --> Q
 G -- Yes --> H[Curated Parquet by run and month]
 H --> I[Destination report]
```

In [ ]:
from pathlib import Path
import json, sys
for candidate in (Path.cwd(), *Path.cwd().parents):
    if (candidate / 'training_lib.py').is_file():
        ROOT = candidate; break
else: raise FileNotFoundError('Launch Jupyter in the workshop folder')
sys.path.insert(0, str(ROOT))
from training_lib import create_spark
from project.pipeline import (load_run_config, read_source_batch, read_references,
    parse_and_classify, enrich_completed, run_pipeline, write_combined_report)
spark = create_spark('guided-ecommerce-pipeline')
spark.version, spark.sparkContext.master, spark.conf.get('spark.sql.adaptive.enabled')

## 1. Inspect the three configured source batches — 10 minutes

In [ ]:
configs = load_run_config(ROOT / 'project')
configs

## 2. Walk through the clean CSV run — 35 minutes

In [ ]:
raw_1 = read_source_batch(spark, 'run-001', ROOT / 'project')
raw_1.printSchema(); raw_1.show(20, truncate=False)
unique_1, valid_1, quality_rejected_1 = parse_and_classify(raw_1)
print('raw', raw_1.count(), 'unique', unique_1.count(), 'valid', valid_1.count(), 'rejected', quality_rejected_1.count())
valid_1.select('transaction_id', 'order_date', 'qty', 'price', 'line_total', 'status').show()

In [ ]:
customers, products = read_references(spark, ROOT / 'project')
completed_1, accepted_1, reference_rejected_1 = enrich_completed(valid_1, customers, products)
accepted_1.select('transaction_id', 'customer_name', 'product_name', 'category', 'line_total').show(truncate=False)
assert completed_1.count() == 10 and reference_rejected_1.count() == 0

**Checkpoint:** Why broadcast the references? Why is `COMPLETED` a business rule rather than a quality rule?

In [ ]:
metrics_1, report_1 = run_pipeline(spark, 'run-001', ROOT / 'project')
from IPython.display import HTML, display
display(metrics_1); display(HTML(report_1.read_text(encoding='utf-8')))

## 3. Run the dirty JSON batch and inspect both rejection stages — 35 minutes

In [ ]:
raw_2 = read_source_batch(spark, 'run-002', ROOT / 'project')
unique_2, valid_2, quality_rejected_2 = parse_and_classify(raw_2)
quality_rejected_2.select('transaction_id', 'reject_reason').show(truncate=False)
completed_2, accepted_2, reference_rejected_2 = enrich_completed(valid_2, customers, products)
reference_rejected_2.select('transaction_id', 'customer_id', 'product_id', 'reject_reason').show(truncate=False)

In [ ]:
metrics_2, report_2 = run_pipeline(spark, 'run-002', ROOT / 'project')
rejected_path = ROOT / 'project' / 'destination' / 'rejected_records' / 'run_id=run-002'
spark.read.parquet(str(rejected_path)).select('transaction_id', 'reject_stage', 'reject_reason').orderBy('transaction_id').show(truncate=False)
display(metrics_2); display(HTML(report_2.read_text(encoding='utf-8')))

## 4. Process corrected records and new sales from TSV — 25 minutes

In [ ]:
raw_3 = read_source_batch(spark, 'run-003', ROOT / 'project')
raw_3.show(20, truncate=False)
metrics_3, report_3 = run_pipeline(spark, 'run-003', ROOT / 'project')
display(metrics_3); display(HTML(report_3.read_text(encoding='utf-8')))

## 5. Replay run-002 and prove that its partition is replaced — 15 minutes

In [ ]:
curated_base = ROOT / 'project' / 'destination' / 'curated_sales'
before = spark.read.option('basePath', str(curated_base)).parquet(str(curated_base)).count()
replay_metrics, _ = run_pipeline(spark, 'run-002', ROOT / 'project', make_report=False)
after = spark.read.option('basePath', str(curated_base)).parquet(str(curated_base)).count()
print({'before': before, 'after': after, 'stable': before == after})
assert before == after == 21

## 6. Create the combined visual report from destination data — 15 minutes

In [ ]:
all_report = write_combined_report(spark, ROOT / 'project')
display(HTML(all_report.read_text(encoding='utf-8')))
destination = spark.read.option('basePath', str(curated_base)).parquet(str(curated_base))
destination.groupBy('run_id', 'order_month').count().orderBy('run_id').show()

## 7. Validate audit metrics and inspect the physical plan — 15 minutes

In [ ]:
expected = json.loads((ROOT / 'solutions' / 'project_expected_metrics.json').read_text())
for run_id, actual in [('run-001', metrics_1), ('run-002', metrics_2), ('run-003', metrics_3)]:
    for key, value in expected[run_id].items(): assert actual[key] == value, (run_id, key, actual[key], value)
print('PASS: all run audits match expected metrics')
from pyspark.sql import functions as F
destination.groupBy('category').agg(F.sum('line_total').alias('revenue')).explain('formatted')

## Debrief

The source format changed while the normalized contract stayed constant. Invalid rows were retained with reasons. Small dimensions used broadcast joins, while aggregation still shuffled. Each run partition is replay-safe, and every report was generated after reading destination Parquet.